# Monte Carlo simulator for prop firm challenges

This notebook implements, step by step, the same Monte Carlo framework (block bootstrap) presented in the theory part of the video: given the returns of a strategy, we estimate the probability of passing a prop firm challenge, the expected value per attempt, the cost of getting a funded account, and the probability of bankroll ruin.

## 1. Strategy returns

We generate the returns of a strategy with **known statistics**: hourly log returns drawn from a normal distribution whose annualized mean return and volatility we choose. Because the true parameters are known, the true Sharpe ratio is known too (`annual return / annual volatility`), and we can compare it with the Sharpe measured on the generated sample.

The **Sharpe ratio** measures return per unit of risk: mean return divided by the standard deviation of returns, annualized by multiplying by `sqrt(bars per year)` (risk-free rate set to 0).

The sample Sharpe is not equal to the true one: with a short sample (3 years here) the estimation error of an annualized Sharpe ratio is large, roughly ±0.7 at a true Sharpe of 1. Increase `N_YEARS` to see the sample values converge to the true ones.

To use your own strategy instead, replace `log_returns` with any `pd.Series` of bar log returns: the rest of the notebook works unchanged.

In [1]:
# pip install numpy pandas plotly
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_dark"

BARS_PER_DAY = 23          # hourly bars in a trading day
DAYS_PER_YEAR = 252
BARS_PER_YEAR = BARS_PER_DAY * DAYS_PER_YEAR

# Known statistics of the simulated strategy (annualized)
ANNUAL_RETURN = 0.10
ANNUAL_VOL = 0.10
N_YEARS = 3


def line_chart(series, title, xlabel, ylabel, hlines=(), points=(), legend=True, fig=None, row=None, col=None):
    """Line chart of one or more series, with optional horizontal levels and marked points."""
    fig = fig or go.Figure()
    pos = dict(row=row, col=col)
    for name, y in series.items():
        fig.add_trace(go.Scatter(y=y, name=name, mode="lines", line=dict(width=1.5), showlegend=legend), **pos)
    for y, name, color in hlines:
        fig.add_hline(y=y, line=dict(color=color, dash="dash"), annotation_text=name, **pos)
    for x, y, name, color in points:
        fig.add_trace(go.Scatter(x=[x], y=[y], name=name, mode="markers", marker=dict(color=color, size=10)), **pos)
    fig.update_layout(title=title, height=500, legend=dict(orientation="h"))
    fig.update_xaxes(title=xlabel, **pos)
    fig.update_yaxes(title=ylabel, **pos)
    return fig


def sharpe_ratio(r, periods_per_year):
    """Annualized Sharpe ratio (risk-free rate = 0)."""
    return np.sqrt(periods_per_year) * r.mean() / r.std()


# Per-bar parameters: the mean scales with time, the volatility with its square root
mu_bar = ANNUAL_RETURN / BARS_PER_YEAR
sigma_bar = ANNUAL_VOL / np.sqrt(BARS_PER_YEAR)

rng_data = np.random.default_rng(42)
log_returns = pd.Series(rng_data.normal(mu_bar, sigma_bar, N_YEARS * BARS_PER_YEAR), name="log_return")

print(f"True   : return {ANNUAL_RETURN:.1%}  vol {ANNUAL_VOL:.1%}  Sharpe {ANNUAL_RETURN / ANNUAL_VOL:.2f}")
print(f"Sample : return {log_returns.mean() * BARS_PER_YEAR:.1%}  "
      f"vol {log_returns.std() * np.sqrt(BARS_PER_YEAR):.1%}  "
      f"Sharpe {sharpe_ratio(log_returns, BARS_PER_YEAR):.2f}")

line_chart({"equity": np.exp(log_returns.cumsum())}, "Equity curve of the simulated strategy",
           "bars", "equity (base 1.0)", legend=False).show()

True   : return 10.0%  vol 10.0%  Sharpe 1.00
Sample : return 7.3%  vol 10.0%  Sharpe 0.73


## 2. Challenge parameters

- `tp` / `sl`: profit target and cumulative stop loss, as a % of capital
- `daily_sl`: maximum allowed daily loss
- `bars_per_day` / `bars_per_month`: how many bars make up a day / a month at your data's frequency (here hourly: 23 and 21 × 23)
- `n_bars`: maximum simulated horizon (max challenge duration)
- `challenge_cost`, `fixed_cost_month`, `payout_pct`, `bankroll`: cost and payout structure
- `block_size`: length of the blocks sampled in the bootstrap

> Note: we use hourly bars, so the daily loss limit is tested bar by bar within each day, capturing intraday excursions. `bars_per_day` must match the frequency of your returns.

In [2]:
params = {
    "tp" : 0.15,
    "sl" : 0.10,
    "daily_sl" : 0.05,
    "trailing_sl" : False,
    "bars_per_day" : BARS_PER_DAY,
    "days_per_month" : 21,
    "n_days" : 252,
    "capital" : 100_000,
    "challenge_cost" : 650,
    "fixed_cost_month" : 15,
    "payout_pct" : 0.80,
    "bankroll" : 10000,
    "block_size" : 1,
}

params["bars_per_month"] = params["days_per_month"] * params["bars_per_day"]
params["n_bars"] = params["n_days"] * params["bars_per_day"]

leverage_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

## 3. Block bootstrap: why resample in blocks

Instead of assuming a theoretical distribution for the simulations, we resample **contiguous blocks** of the strategy's returns. With real history this preserves the autocorrelation and market regimes (good/bad days that cluster together) that independent sampling would destroy. Each generated path is a plausible alternative history, built by reshuffling pieces of the strategy.

The returns generated above are independent by construction, so there is no clustering to preserve and `block_size = 1` is enough here. With a real strategy, try larger block sizes.

In [3]:
def sample_bootstrap_path(returns_simple, p, rng):
    T = len(returns_simple)
    block_size = min(T, max(1, p["block_size"]))
    n_blocks = int(np.ceil(p["n_bars"] / block_size))
    max_start_idx = T - block_size

    start_idx = rng.integers(0, max_start_idx + 1, size=n_blocks)
    r = np.concatenate(
        [returns_simple[s : s + block_size] for s in start_idx]
    )[: p["n_bars"]]
    return r


returns_simple = (np.exp(log_returns.dropna()) - 1).values  # log -> simple return

rng_demo = np.random.default_rng()
paths = {f"path {k + 1}": (1 + sample_bootstrap_path(returns_simple, params, rng_demo)).cumprod() for k in range(15)}
line_chart(paths, "15 alternative histories sampled via block bootstrap", "bars", "equity (base 1.0)", legend=False).show()

## 4. A single path: where TP, cumulative SL and daily loss trigger

Before running thousands of simulations, let's look at what happens on **a single path**: we compute the cumulative equity, test the take profit, the cumulative stop loss, and — day by day — the return accumulated since each day's open against the daily loss limit.

In [4]:
rng_single = np.random.default_rng()
r = sample_bootstrap_path(returns_simple, params, rng_single)
cum = (1 + r).cumprod()

tp_level = 1 + params["tp"]
sl_level = 1 - params["sl"]
hit_sl_cum = cum <= sl_level

n_days = params["n_bars"] // params["bars_per_day"]
daily_r = np.zeros(params["n_bars"])
for day in range(n_days):
    start = day * params["bars_per_day"]
    end = (day + 1) * params["bars_per_day"]
    daily_r[start:end] = np.cumprod(1 + r[start:end]) - 1
hit_sl_daily = daily_r <= -params["daily_sl"]

hit_sl = hit_sl_cum | hit_sl_daily
hit_tp = cum >= tp_level
tp_idx = np.argmax(hit_tp) if hit_tp.any() else None
sl_idx = np.argmax(hit_sl) if hit_sl.any() else None

points = []
if tp_idx is not None:
    points.append((tp_idx, cum[tp_idx], "TP hit", "green"))
if sl_idx is not None:
    points.append((sl_idx, cum[sl_idx], "SL hit", "red"))

line_chart({"simulated equity": cum}, "A single simulated path: TP vs SL", "bars", "equity (base 1.0)",
           hlines=[(tp_level, "take profit", "green"), (sl_level, "cumulative stop loss", "red")],
           points=points).show()

print("tp_idx:", tp_idx, " sl_idx:", sl_idx)

tp_idx: None  sl_idx: None


## 5. The full simulation function

We put all the pieces together into a function that, given the strategy returns, the leverage and the challenge parameters, runs `n_sim` independent simulations and returns the aggregate metrics: probability of success, expected value, cost to pass, efficiency, probability of ruin.

Key points of the logic:
- returns are converted from log to simple **before** applying leverage (leverage scales the real P&L, not the log return)
- the daily loss limit test uses the return accumulated **since each day's open**, bar by bar — not just the closing bar
- once the challenge is passed, the "funded" account continues under its own rules (cumulative SL and daily SL recomputed from its own starting equity)
- the payout is estimated around the funded phase's equity peak, with a roughly one-month random offset (payouts are monthly, not instantaneous at the peak)

In [5]:
def run_simulation(log_returns, leverage, p, n_sim, seed=None):
    returns = np.exp(log_returns.dropna()) - 1  # log -> simple
    T = len(returns)

    block_size = min(T, max(1, p["block_size"]))
    n_blocks = int(np.ceil(p["n_bars"] / block_size))
    max_start_idx = T - block_size

    leveraged_returns = returns * leverage
    rng = np.random.default_rng(seed)

    result_list, bars_to_tp_list, bars_to_end_list = [], [], []
    final_return_list, loss_type_list = [], []
    max_drawdown_list, max_daily_drawdown_list = [], []

    for _ in range(n_sim):
        start_idx = rng.integers(0, max_start_idx + 1, size=n_blocks)
        r = np.concatenate(
            [leveraged_returns.values[s : s + block_size] for s in start_idx]
        )[: p["n_bars"]]

        cum = (1 + r).cumprod()
        tp_level = 1 + p["tp"]
        sl_level = 1 - p["sl"]

        if p["trailing_sl"]:
            drawdown = cum / np.maximum.accumulate(cum) - 1
            hit_sl_cum = drawdown <= -p["sl"]
        else:
            hit_sl_cum = cum <= sl_level

        n_days = p["n_bars"] // p["bars_per_day"]
        daily_r = np.zeros(p["n_bars"])
        for day in range(n_days):
            start = day * p["bars_per_day"]
            end = (day + 1) * p["bars_per_day"]
            daily_r[start:end] = np.cumprod(1 + r[start:end]) - 1
        hit_sl_daily = daily_r <= -p["daily_sl"]

        hit_sl = hit_sl_cum | hit_sl_daily
        hit_tp = cum >= tp_level

        tp_idx = np.argmax(hit_tp) if hit_tp.any() else None
        sl_idx = np.argmax(hit_sl) if hit_sl.any() else None

        sl_type = None
        if sl_idx is not None:
            if hit_sl_daily[sl_idx]:
                sl_type = "daily"
            elif hit_sl_cum[sl_idx]:
                sl_type = "cumulative"

        if tp_idx is not None and (sl_idx is None or tp_idx <= sl_idx):
            outcome = 1

            cum_after = cum[tp_idx:]
            daily_r_after = daily_r[tp_idx:]

            if p["trailing_sl"]:
                drawdown_after = cum_after / np.maximum.accumulate(cum_after) - 1
                hit_sl_cum_after = drawdown_after <= -p["sl"]
            else:
                hit_sl_cum_after = cum_after <= sl_level

            hit_sl_daily_after = daily_r_after <= -p["daily_sl"]
            hit_sl_after = hit_sl_cum_after | hit_sl_daily_after

            if hit_sl_after.any():
                sl_after_idx = np.argmax(hit_sl_after)
                end_idx = tp_idx + sl_after_idx
            else:
                end_idx = len(cum) - 1

            bars_to_tp = tp_idx
            bars_to_end = end_idx

            cum_funded = cum[tp_idx : end_idx + 1]
            max_idx = np.argmax(cum_funded) + tp_idx

            real_max_idx = (
                max_idx
                + rng.integers(0, p["bars_per_month"] + 1)
                - p["bars_per_month"] // 2
            )
            min_payout_idx = tp_idx + p["bars_per_month"]

            if min_payout_idx <= real_max_idx <= end_idx:
                final_return = cum[real_max_idx] - 1
            else:
                prev_real_max_idx = real_max_idx - p["bars_per_month"]
                if min_payout_idx <= prev_real_max_idx <= end_idx:
                    final_return = cum[prev_real_max_idx] - 1
                else:
                    final_return = 0

            daily_r_slice = daily_r[: end_idx + 1]
            cum_slice = cum[: end_idx + 1]

        elif sl_idx is not None:
            outcome = -1
            bars_to_tp = np.nan
            bars_to_end = sl_idx
            final_return = np.nan

            daily_r_slice = daily_r[: sl_idx + 1]
            cum_slice = cum[: sl_idx + 1]

        else:
            outcome = 0
            bars_to_tp = np.nan
            bars_to_end = len(cum) - 1
            final_return = np.nan

            daily_r_slice = daily_r
            cum_slice = cum

        drawdown_slice = cum_slice / np.maximum.accumulate(cum_slice) - 1
        max_drawdown_list.append(drawdown_slice.min())
        max_daily_drawdown_list.append(daily_r_slice.min())

        result_list.append(outcome)
        bars_to_tp_list.append(bars_to_tp)
        bars_to_end_list.append(bars_to_end)
        final_return_list.append(final_return)
        loss_type_list.append(sl_type)

    result_list = np.array(result_list)
    bars_to_tp_list = np.array(bars_to_tp_list, dtype=float)
    bars_to_end_list = np.array(bars_to_end_list, dtype=float)
    final_return_list = np.array(final_return_list, dtype=float)
    loss_type_list = np.array(loss_type_list)
    max_drawdown_list = np.array(max_drawdown_list, dtype=float)
    max_daily_drawdown_list = np.array(max_daily_drawdown_list, dtype=float)

    success_mask = result_list == 1
    loss_mask = result_list == -1

    loss_daily_rate = np.mean(loss_mask & (loss_type_list == "daily"))
    loss_cum_rate = np.mean(loss_mask & (loss_type_list == "cumulative"))

    success_rate = success_mask.mean()
    loss_rate = loss_mask.mean()
    no_event_rate = (result_list == 0).mean()

    avg_bars_to_tp = np.nanmean(bars_to_tp_list[success_mask]) if success_mask.any() else np.nan
    avg_bars_to_end = np.nanmean(bars_to_end_list)

    net_profit = np.maximum((final_return_list - p["tp"]) * p["capital"], 0)
    net_payout = net_profit * p["payout_pct"]

    monthly_cost = (bars_to_end_list / p["bars_per_month"]) * p["fixed_cost_month"]
    avg_monthly_cost = np.nanmean(monthly_cost)

    expected_payout = np.nanmean(net_payout[success_mask]) if success_mask.any() else 0

    # EV: the challenge fee is assumed refunded on passing -> weighted by (1-success_rate)
    # avg_monthly_cost is already an unconditional expectation, it should not be reweighted
    ev = success_rate * expected_payout - (1 - success_rate) * p["challenge_cost"] - avg_monthly_cost

    # cost_to_pass: expected gross outlay to get a funded account, refunded or not
    cost_to_pass = (p["challenge_cost"] + avg_monthly_cost) / success_rate if success_rate > 0 else 0
    efficiency = ev / cost_to_pass if cost_to_pass > 0 else 0
    prob_ruin = (1 - success_rate) ** (p["bankroll"] / (p["challenge_cost"] + avg_monthly_cost))

    return {
        "leverage": leverage,
        "success_rate": success_rate,
        "loss_rate": loss_rate,
        "no_event_rate": no_event_rate,
        "loss_daily_rate": loss_daily_rate,
        "loss_cumulative_rate": loss_cum_rate,
        "avg_max_drawdown": round(max_drawdown_list.mean(), 3),
        "avg_max_daily_drawdown": round(max_daily_drawdown_list.mean(), 3),
        "avg_days_to_tp": int(avg_bars_to_tp / p["bars_per_day"]) if success_mask.any() else np.nan,
        "avg_days_to_end": int(avg_bars_to_end / p["bars_per_day"]),
        "avg_final_return": round(np.nanmean(final_return_list), 3) if success_mask.any() else 0,
        "avg_net_payout": int(np.nanmean(net_payout)) if success_mask.any() else 0,
        "expected_value": round(ev, 1),
        "cost_to_pass": round(cost_to_pass, 1) if success_mask.any() else np.nan,
        "efficiency": round(efficiency, 3) if success_mask.any() else np.nan,
        "prob_ruin": round(prob_ruin, 4),
    }

## 6. Running the Monte Carlo for a single leverage

In [6]:
metrics = run_simulation(log_returns, leverage=1, p=params, n_sim=1000, seed=42)
pd.Series(metrics)

leverage                     1.0000
success_rate                 0.3690
loss_rate                    0.1190
no_event_rate                0.5120
loss_daily_rate              0.0000
loss_cumulative_rate         0.1190
avg_max_drawdown            -0.0900
avg_max_daily_drawdown      -0.0180
avg_days_to_tp             162.0000
avg_days_to_end            239.0000
avg_final_return             0.1530
avg_net_payout            4029.0000
expected_value             905.7000
cost_to_pass              2225.1000
efficiency                   0.4070
prob_ruin                    0.0037
dtype: float64

## 7. The effect of leverage

We repeat the simulation for different leverage levels on the same returns, to see how the probability of success and expected value move. More leverage means reaching the target faster, but also approaching the daily loss limit faster — the net effect isn't obvious and needs to be measured, not assumed.

In [7]:
rows = [run_simulation(log_returns, lev, params, n_sim=1000, seed=42) for lev in leverage_list]
results_df = pd.DataFrame(rows, index = leverage_list)
results_df.transpose()

,1,2,3,4,5,6,7,8,9,10
leverage,1.0000,2.000,3.0000,4.0000,5.0000,6.0000,7.0000,8.0000,9.0000,10.0000
success_rate,0.3690,0.612,0.5060,0.3510,0.2800,0.2460,0.2260,0.2060,0.1980,0.2030
loss_rate,0.1190,0.349,0.4940,0.6490,0.7200,0.7540,0.7740,0.7940,0.8020,0.7970
no_event_rate,0.5120,0.039,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
loss_daily_rate,0.0000,0.001,0.1600,0.4620,0.6190,0.6930,0.7490,0.7750,0.7930,0.7910
loss_cumulative_rate,0.1190,0.348,0.3340,0.1870,0.1010,0.0610,0.0250,0.0190,0.0090,0.0060
avg_max_drawdown,-0.0900,-0.151,-0.1600,-0.1140,-0.0950,-0.0870,-0.0820,-0.0800,-0.0820,-0.0810
avg_max_daily_drawdown,-0.0180,-0.034,-0.0470,-0.0510,-0.0530,-0.0540,-0.0550,-0.0560,-0.0560,-0.0570
avg_days_to_tp,162.0000,87.000,40.0000,19.0000,10.0000,7.0000,4.0000,3.0000,2.0000,2.0000
avg_days_to_end,239.0000,187.000,102.0000,27.0000,12.0000,6.0000,4.0000,3.0000,2.0000,2.0000


In [8]:
panels = [  # (title, y label, columns to plot, zero line)
    ("Probability of pass vs leverage", "success rate", ["success_rate"], False),
    ("Avg days to TP vs leverage", "days", ["avg_days_to_tp"], False),
    ("Avg days to account close vs leverage", "days", ["avg_days_to_end"], False),
    ("Expected value vs leverage", "EV ($)", ["expected_value"], True),
    ("Failure cause vs leverage", "rate", ["loss_daily_rate", "loss_cumulative_rate"], False),
    ("Probability of ruin vs leverage", "prob. of ruin", ["prob_ruin"], False),
]

fig = make_subplots(rows=2, cols=3, subplot_titles=[p[0] for p in panels])
for k, (title, ylabel, cols, zero_line) in enumerate(panels):
    r, c = divmod(k, 3)
    for col in cols:
        fig.add_trace(go.Scatter(x=results_df["leverage"], y=results_df[col], name=col, mode="lines+markers",
                                 showlegend=len(cols) > 1), row=r + 1, col=c + 1)
    if zero_line:
        fig.add_hline(y=0, line=dict(color="grey", dash="dot"), row=r + 1, col=c + 1)
    fig.update_xaxes(title="leverage", row=r + 1, col=c + 1)
    fig.update_yaxes(title=ylabel, row=r + 1, col=c + 1)
fig.update_layout(height=800)
fig.show()